<a href="https://colab.research.google.com/github/ProductPriceTrackerOrg/data-science/blob/main/notebooks/product-category/02_model_building_and_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Multi-class Clasification for Product category Prediction**

## Import the packages and libraries

This cell imports all necessary libraries for data processing, model building,
training, and evaluation. We need pandas for data handling, sklearn for metrics
and data splitting, torch for the ANN model, transformers for BERT, and
sentence-transformers for embeddings.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import warnings
warnings.filterwarnings('ignore')

## Load and Explore Dataset

In [2]:
df = pd.read_csv('/content/products_summary_filtered.csv')

print("Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nFirst 5 rows:")
print(df.head())

print(f"\nMissing values:")
print(df.isnull().sum())

print(f"\nNumber of unique categories: {df['category'].nunique()}")
print(f"\nCategory distribution:")
print(df['category'].value_counts())

Dataset Overview:
Shape: (21440, 2)
Columns: ['product_title', 'category']

First 5 rows:
                                       product_title  \
0                     Huawei Watch GT 4 46MM – Black   
1                                   Xiaomi Mi Band 8   
2                  TP-Link M7200 4G LTE Mobile Wi-Fi   
3  Apple MGN63 13.3-inch MacBook Air M1 Chip with...   
4                                         JBL Flip 6   

                      category  
0  Smart Watches & Accessories  
1  Smart Watches & Accessories  
2                   Networking  
3                      Laptops  
4                     Speakers  

Missing values:
product_title    0
category         0
dtype: int64

Number of unique categories: 28

Category distribution:
category
Cases & Screen Protectors             5123
Chargers & Power Banks                2412
Cables & Adapters                     2382
Mobile Phones                         1544
Headphones & Earbuds                  1334
Smart Watches & Accessorie

## Data Preprocessing and Label Encoding

In [3]:
# Create label encoder for categories
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['category'])

print("Label Encoding Mapping:")
for i, category in enumerate(label_encoder.classes_):
    print(f"{category} -> {i}")

# Split data into train and test sets (80-20 split, stratified)
X = df['product_title'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nData Split:")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Number of classes: {len(label_encoder.classes_)}")

Label Encoding Mapping:
Bags, Sleeves & Backpacks -> 0
CPUs -> 1
Cables & Adapters -> 2
Camera Accessories -> 3
Cameras & Drones -> 4
Car Accessories -> 5
Cases & Screen Protectors -> 6
Chargers & Power Banks -> 7
Gaming Peripherals -> 8
Graphic Cards -> 9
Headphones & Earbuds -> 10
Health & Personal Care Electronics -> 11
Keyboards -> 12
Laptops -> 13
Memory -> 14
Mice -> 15
Mobile Phones -> 16
Monitors -> 17
Motherboards -> 18
Networking -> 19
Power Supplies & PC Cooling -> 20
Printers & Scanners -> 21
Smart Home & Office Accessories -> 22
Smart Watches & Accessories -> 23
Speakers -> 24
Storage -> 25
Tablets -> 26
Webcams & Microphones -> 27

Data Split:
Training samples: 17152
Testing samples: 4288
Number of classes: 28


## Generate SBERT Embeddings for ANN Model

- This cell generates sentence embeddings using the pre-trained SBERT model.
- These 384-dimensional embeddings will serve as input features for our ANN.
- The all-MiniLM-L6-v2 model is lightweight yet effective for text classification.

In [4]:
# Load pre-trained sentence transformer
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for training and test sets
print("Generating training embeddings...")
X_train_embeddings = sbert_model.encode(X_train, show_progress_bar=True)

print("Generating test embeddings...")
X_test_embeddings = sbert_model.encode(X_test, show_progress_bar=True)

print(f"Embedding shape: {X_train_embeddings.shape}")
print(f"Embedding dimensions: {X_train_embeddings.shape[1]}")

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_embeddings)
X_test_tensor = torch.FloatTensor(X_test_embeddings)
y_train_tensor = torch.LongTensor(y_train)
y_test_tensor = torch.LongTensor(y_test)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Generating training embeddings...


Batches:   0%|          | 0/536 [00:00<?, ?it/s]

Generating test embeddings...


Batches:   0%|          | 0/134 [00:00<?, ?it/s]

Embedding shape: (17152, 384)
Embedding dimensions: 384


## Define ANN Model Architecture

- This cell defines the Artificial Neural Network architecture.
- The network has two hidden layers with ReLU activations and dropout
for regularization.
- The architecture follows the specification:
384 -> 256 -> 128 -> 28 (number of categories)

In [28]:
class ProductCategoryANN(nn.Module):
    def __init__(self, input_dim=384, hidden1_dim=256, hidden2_dim=128, num_classes=28):
        super(ProductCategoryANN, self).__init__()

        # Define layers
        self.fc1 = nn.Linear(input_dim, hidden1_dim)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.5)

        self.fc2 = nn.Linear(hidden1_dim, hidden2_dim)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.5)

        self.fc3 = nn.Linear(hidden2_dim, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        return x

# Initialize the model
ann_model = ProductCategoryANN()
print("ANN Model Architecture:")
print(ann_model)

# Count parameters
total_params = sum(p.numel() for p in ann_model.parameters())
trainable_params = sum(p.numel() for p in ann_model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

ANN Model Architecture:
ProductCategoryANN(
  (fc1): Linear(in_features=384, out_features=256, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.5, inplace=False)
  (fc3): Linear(in_features=128, out_features=28, bias=True)
)

Total parameters: 135,068
Trainable parameters: 135,068


## Train ANN Model

- This cell implements the training loop for the ANN model.
- We use CrossEntropyLoss and Adam optimizer with a standard training procedure.
- The model is trained for multiple epochs with batch processing.

In [29]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Move model to device
ann_model = ann_model.to(device)

# Create data loaders
batch_size = 16
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ann_model.parameters(), lr=0.0005)

# Training loop
num_epochs = 50
print(f"\nTraining ANN model for {num_epochs} epochs...")

for epoch in range(num_epochs):
    ann_model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, targets) in enumerate(train_loader):
        data, targets = data.to(device), targets.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = ann_model(data)
        loss = criterion(outputs, targets)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

    accuracy = 100 * correct / total
    avg_loss = total_loss / len(train_loader)

    if (epoch + 1) % 1 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')

Using device: cuda

Training ANN model for 50 epochs...
Epoch [1/50], Loss: 1.3347, Accuracy: 64.69%
Epoch [2/50], Loss: 0.5985, Accuracy: 83.74%
Epoch [3/50], Loss: 0.4370, Accuracy: 88.54%
Epoch [4/50], Loss: 0.3572, Accuracy: 90.56%
Epoch [5/50], Loss: 0.3206, Accuracy: 91.55%
Epoch [6/50], Loss: 0.2913, Accuracy: 92.14%
Epoch [7/50], Loss: 0.2729, Accuracy: 92.80%
Epoch [8/50], Loss: 0.2521, Accuracy: 93.05%
Epoch [9/50], Loss: 0.2399, Accuracy: 93.31%
Epoch [10/50], Loss: 0.2236, Accuracy: 93.72%
Epoch [11/50], Loss: 0.2104, Accuracy: 94.01%
Epoch [12/50], Loss: 0.2018, Accuracy: 93.99%
Epoch [13/50], Loss: 0.1975, Accuracy: 94.43%
Epoch [14/50], Loss: 0.1896, Accuracy: 94.72%
Epoch [15/50], Loss: 0.1749, Accuracy: 94.88%
Epoch [16/50], Loss: 0.1707, Accuracy: 94.87%
Epoch [17/50], Loss: 0.1691, Accuracy: 94.79%
Epoch [18/50], Loss: 0.1620, Accuracy: 95.32%
Epoch [19/50], Loss: 0.1573, Accuracy: 95.14%
Epoch [20/50], Loss: 0.1541, Accuracy: 95.22%
Epoch [21/50], Loss: 0.1479, Accu

## Evaluate ANN Model

- This cell evaluates the trained ANN model on the test set.
- We calculate accuracy, precision, recall, and F1-score using sklearn metrics.
- All predictions are made in evaluation mode with no gradient computation.

In [30]:
def evaluate_model(model, data_loader, device, model_name):
    """
    Evaluate a PyTorch model and return comprehensive metrics.
    """
    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for data, targets in data_loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)

            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    # Calculate metrics
    accuracy = accuracy_score(all_targets, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets, all_predictions, average='weighted', zero_division=0
    )

    print(f"\n{model_name} Evaluation Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Weighted Precision: {precision:.4f}")
    print(f"Weighted Recall: {recall:.4f}")
    print(f"Weighted F1-Score: {f1:.4f}")

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Evaluate ANN model
print("\nEvaluating ANN model...")
print("-" * 50)

print("Evaluation Results for Training Data:")
ann_results_train = evaluate_model(ann_model, train_loader, device, "ANN Model")

print("\nEvaluation Results for Test Data:")
ann_results = evaluate_model(ann_model, test_loader, device, "ANN Model")


Evaluating ANN model...
--------------------------------------------------
Evaluation Results for Training Data:

ANN Model Evaluation Results:
Accuracy: 0.9782
Weighted Precision: 0.9790
Weighted Recall: 0.9782
Weighted F1-Score: 0.9776

Evaluation Results for Test Data:

ANN Model Evaluation Results:
Accuracy: 0.9429
Weighted Precision: 0.9432
Weighted Recall: 0.9429
Weighted F1-Score: 0.9423


## Prepare Data for Transformer Model (Efficient Pre-tokenization)

This cell prepares the text data for the DistilBERT model using efficient
pre-tokenization. Instead of tokenizing on-the-fly, we pre-tokenize the entire
dataset at once using batch processing, which is much faster and follows
Hugging Face best practices.

In [32]:
from datasets import Dataset

# Load tokenizer for DistilBERT
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    """
    Tokenization function for batch processing.
    This function will be applied to the entire dataset at once.
    """
    return tokenizer(
        examples['product_title'],
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors=None  # Returns lists, not tensors for efficiency
    )

print("Creating Hugging Face datasets...")

# Create train dataset
train_data = {
    'product_title': X_train.tolist(),
    'labels': y_train.tolist()
}
train_dataset = Dataset.from_dict(train_data)

# Create test dataset
test_data = {
    'product_title': X_test.tolist(),
    'labels': y_test.tolist()
}
test_dataset = Dataset.from_dict(test_data)

print("Pre-tokenizing datasets (this is much faster than on-the-fly tokenization)...")

# Pre-tokenize both datasets using batch processing
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    desc="Tokenizing train dataset"
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    desc="Tokenizing test dataset"
)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"\nTransformer datasets created and pre-tokenized:")
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Test tokenization
sample = train_dataset[0]
print(f"\nSample tokenized data shape:")
print(f"Input IDs: {sample['input_ids'].shape}")
print(f"Attention Mask: {sample['attention_mask'].shape}")
print(f"Label: {sample['labels']}")
print("Pre-tokenization completed - much more efficient!")

Creating Hugging Face datasets...
Pre-tokenizing datasets (this is much faster than on-the-fly tokenization)...


Tokenizing train dataset:   0%|          | 0/17152 [00:00<?, ? examples/s]

Tokenizing test dataset:   0%|          | 0/4288 [00:00<?, ? examples/s]


Transformer datasets created and pre-tokenized:
Training samples: 17152
Test samples: 4288

Sample tokenized data shape:
Input IDs: torch.Size([128])
Attention Mask: torch.Size([128])
Label: 7
Pre-tokenization completed - much more efficient!


## Define and Initialize DistilBERT Model

This cell loads the pre-trained DistilBERT model and configures it for
our classification task. The model is initialized with 28 output classes
corresponding to our product categories.

In [33]:
# Load pre-trained DistilBERT model for sequence classification
bert_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=28
)

print("DistilBERT Model loaded successfully!")
print(f"Model type: {type(bert_model)}")

# Count parameters in BERT model
total_params = sum(p.numel() for p in bert_model.parameters())
trainable_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"\nDistilBERT parameters:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBERT Model loaded successfully!
Model type: <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'>

DistilBERT parameters:
Total parameters: 66,975,004
Trainable parameters: 66,975,004


## Configure Training Arguments for DistilBERT

This cell sets up the training configuration for the DistilBERT model.
We define training arguments including batch size, learning rate, number of epochs,
and evaluation strategy using Hugging Face's TrainingArguments.

In [34]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    # evaluation_strategy="epoch",
    # save_strategy="epoch",
    # load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to=None  # Disable wandb logging
)


print("Training Arguments configured:")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Weight decay: {training_args.weight_decay}")

Training Arguments configured:
Epochs: 3
Batch size: 16
Learning rate: 5e-05
Weight decay: 0.01


## Train DistilBERT Model

This cell trains the DistilBERT model using the Hugging Face Trainer API.
The Trainer handles the training loop, evaluation, and model checkpointing
automatically. We use a data collator for efficient batch processing.

In [35]:
# Define data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Define compute metrics function for evaluation during training
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Initialize trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting DistilBERT training...")

# Train the model
trainer.train()

print("DistilBERT training completed!")

Starting DistilBERT training...


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: fernandonpa-22 (fernandonpa-22-university-of-moratuwa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
100,3.152100
200,2.263200
300,1.270200
400,0.831600
500,0.553800
600,0.413600
700,0.329400
800,0.311900
900,0.312400
1000,0.286300


DistilBERT training completed!


## Evaluate DistilBERT Model

This cell evaluates the fine-tuned DistilBERT model on the test set.
We use the trainer's evaluate method and also implement manual evaluation
to ensure consistency with our ANN evaluation metrics.

In [36]:
# Evaluate using trainer
eval_results = trainer.evaluate()

print("DistilBERT Evaluation Results (from Trainer):")
for key, value in eval_results.items():
    if key.startswith('eval_'):
        metric_name = key.replace('eval_', '').title()
        print(f"{metric_name}: {value:.4f}")

# Manual evaluation for consistency
bert_model.eval()
all_predictions = []
all_targets = []

print("\nPerforming manual evaluation for consistency...")

with torch.no_grad():
    for batch in DataLoader(test_dataset, batch_size=16):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)

        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

# Calculate metrics manually
bert_accuracy = accuracy_score(all_targets, all_predictions)
bert_precision, bert_recall, bert_f1, _ = precision_recall_fscore_support(
    all_targets, all_predictions, average='weighted', zero_division=0
)

bert_results = {
    'accuracy': bert_accuracy,
    'precision': bert_precision,
    'recall': bert_recall,
    'f1': bert_f1
}

print("\nDistilBERT Manual Evaluation Results:")
print(f"Accuracy: {bert_accuracy:.4f}")
print(f"Weighted Precision: {bert_precision:.4f}")
print(f"Weighted Recall: {bert_recall:.4f}")
print(f"Weighted F1-Score: {bert_f1:.4f}")

DistilBERT Evaluation Results (from Trainer):
Loss: 0.2111
Accuracy: 0.9478
Precision: 0.9480
Recall: 0.9478
F1: 0.9472
Runtime: 15.2426
Samples_Per_Second: 281.3180
Steps_Per_Second: 17.5820

Performing manual evaluation for consistency...

DistilBERT Manual Evaluation Results:
Accuracy: 0.9478
Weighted Precision: 0.9480
Weighted Recall: 0.9478
Weighted F1-Score: 0.9472


## Model Comparison and Final Results

This cell compares the performance of both models and provides a final
recommendation. We compare all metrics side-by-side and determine the
best model based on the highest weighted F1-score.

In [38]:
print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)

print("\nModel Performance Summary:")
print(f"{'Metric':<20} {'ANN Model':<15} {'DistilBERT':<15}")
print("-" * 50)
print(f"{'Accuracy':<20} {ann_results['accuracy']:<15.4f} {bert_results['accuracy']:<15.4f}")
print(f"{'Precision (Weighted)':<20} {ann_results['precision']:<15.4f} {bert_results['precision']:<15.4f}")
print(f"{'Recall (Weighted)':<20} {ann_results['recall']:<15.4f} {bert_results['recall']:<15.4f}")
print(f"{'F1-Score (Weighted)':<20} {ann_results['f1']:<15.4f} {bert_results['f1']:<15.4f}")

# Determine best model based on F1-score
if ann_results['f1'] > bert_results['f1']:
    best_model = "ANN Model"
    performance_diff = ann_results['f1'] - bert_results['f1']
else:
    best_model = "DistilBERT"
    performance_diff = bert_results['f1'] - ann_results['f1']

print(f"\n RECOMMENDATION:")
print(f"Best Model: {best_model}")
print(f"Performance advantage: {performance_diff:.4f} F1-score points")



FINAL MODEL COMPARISON

Model Performance Summary:
Metric               ANN Model       DistilBERT     
--------------------------------------------------
Accuracy             0.9429          0.9478         
Precision (Weighted) 0.9432          0.9480         
Recall (Weighted)    0.9429          0.9478         
F1-Score (Weighted)  0.9423          0.9472         

 RECOMMENDATION:
Best Model: DistilBERT
Performance advantage: 0.0049 F1-score points


## Sample Predictions

In [39]:
def predict_category(text, model_type='both'):
    """
    Predict category for a given product title using specified model(s).
    """
    print(f"\nPredicting category for: '{text}'")

    if model_type in ['ann', 'both']:
        # ANN prediction
        ann_model.eval()
        with torch.no_grad():
            # Generate embedding
            text_embedding = sbert_model.encode([text])
            text_tensor = torch.FloatTensor(text_embedding).to(device)

            # Get prediction
            ann_output = ann_model(text_tensor)
            ann_pred = torch.argmax(ann_output, dim=1).cpu().numpy()[0]
            ann_category = label_encoder.inverse_transform([ann_pred])[0]
            ann_confidence = torch.softmax(ann_output, dim=1).max().item()

            print(f"ANN Prediction: {ann_category} (confidence: {ann_confidence:.3f})")

    if model_type in ['bert', 'both']:
        # BERT prediction
        bert_model.eval()
        with torch.no_grad():
            # Tokenize
            inputs = tokenizer(text, return_tensors='pt', truncation=True,
                             padding='max_length', max_length=128)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Get prediction
            bert_output = bert_model(**inputs)
            bert_pred = torch.argmax(bert_output.logits, dim=1).cpu().numpy()[0]
            bert_category = label_encoder.inverse_transform([bert_pred])[0]
            bert_confidence = torch.softmax(bert_output.logits, dim=1).max().item()

            print(f"BERT Prediction: {bert_category} (confidence: {bert_confidence:.3f})")

# Demo predictions on sample texts
sample_texts = [
    "Apple MacBook Pro 16-inch Laptop",
    "Sony WH-1000XM4 Wireless Headphones",
    "Samsung 4K Ultra HD Smart TV 55 inch"
]

print("\n" + "="*50)
print("SAMPLE PREDICTIONS DEMO")
print("="*50)

for text in sample_texts:
    predict_category(text)

print(f"\n Complete implementation finished!")
print(f"Both models trained and evaluated successfully!")


SAMPLE PREDICTIONS DEMO

Predicting category for: 'Apple MacBook Pro 16-inch Laptop'
ANN Prediction: Laptops (confidence: 1.000)
BERT Prediction: Laptops (confidence: 0.999)

Predicting category for: 'Sony WH-1000XM4 Wireless Headphones'
ANN Prediction: Headphones & Earbuds (confidence: 1.000)
BERT Prediction: Headphones & Earbuds (confidence: 0.997)

Predicting category for: 'Samsung 4K Ultra HD Smart TV 55 inch'
ANN Prediction: Smart Home & Office Accessories (confidence: 0.965)
BERT Prediction: Smart Home & Office Accessories (confidence: 0.991)

 Complete implementation finished!
Both models trained and evaluated successfully!


## Save the Models

In [40]:
# --- Saving the ANN Model ---
print("\nSaving the trained ANN model...")
torch.save(ann_model.state_dict(), 'ann_category_classifier.pth')
print("ANN model saved successfully!")

# --- Saving the Fine-Tuned Transformer Model ---
print("\nSaving the fine-tuned DistilBERT model...")
# The Trainer has already saved the best model during training.
# We can save it again to a specific name for clarity.
trainer.save_model('distilbert_category_classifier')
print("DistilBERT model saved to the 'distilbert_category_classifier' directory.")


Saving the trained ANN model...
ANN model saved successfully!

Saving the fine-tuned DistilBERT model...
DistilBERT model saved to the 'distilbert_category_classifier' directory.


In [41]:
import shutil
from google.colab import files

# Replace "my_folder" with your folder name
shutil.make_archive("/content/distilbert_category_classifier", 'zip', "/content/distilbert_category_classifier")

# Download the zip
files.download("/content/distilbert_category_classifier.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>